# Unit 7 solutions: semantic text embeddings

These reference solutions open Part 2. Unit 3's lexical path called two books similar only when they shared surface **words**; here we match on **meaning** instead. Every vocabulary token carries a pretrained, dense **GloVe** vector -- a point in a 100-dimensional semantic space where nearness reflects meaning -- a book embedding is the **mean** of its keyword tokens' vectors (L2-normalized), a reader embedding is the aggregate of the books they have seen, and retrieval is **brute-force cosine** over the catalog. We first read the geometry directly (cosine nearest words, scoped to the catalog's own vocabulary), build a book and a reader embedding by hand and confirm them against the shipped `book_embedding`, then register `SemanticEmbeddingRetrievalPath` and read its validation scoreboard honestly.

Read the numbers **honestly**, exactly as the lesson does. Measured on this data (seed 0, k=10, 500 val readers), semantic mean-pooling scores hit@10 ~0.102: well above the random floor (~0.012, about 8.5x) but **below** the lexical path (~0.158, about 65% of it), and far below the collaborative item-item CF (~0.252) and matrix-factorization (~0.276) paths. It is **not** a win. The reason is specific and honest: this catalog's latent-taste topics are arbitrary contiguous slices of a general word pool, so GloVe's meaning-geometry is uncorrelated-by-construction with the taste structure; only the 12 genre banks carry GloVe-exploitable meaning. On a real catalog's prose, dense semantics would fare differently -- we teach the *method* honestly, not a staged win. Semantic's real value is **complementarity**: its top-10 overlap with lexical is only ~0.22, so it surfaces meaning-related books lexical misses, and the Challenge measurements show a weaker pooling can rank higher and that adding semantic to the Unit 6 pinned blend lifts both hit *and* coverage.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The `test` holdout stays sealed -- no cell here touches it.


In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PathRegistry,
    PopularityRetrievalPath,
    RandomRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    blend,
    book_embedding,
    load_glove_subset,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

SEED = 0
K = 10        # every scoreboard reports at the top-10 cutoff

# --- the real interaction log (the same substrate as Units 1-6) ----------------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)                      # 2000 books
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")  # {item_id: "space joined token bag"}

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])           # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# The committed, checksum-pinned GloVe subset: a (608, 100) matrix + a token->row index, loaded with
# numpy only (no gensim). One vocabulary word ('starfall') is out of vocabulary: a zero row, listed in
# glove.missing, skipped at pooling.
glove = load_glove_subset()

# Each reader's train-positive history is their `seen` (exclusion) set; their val positives are the
# relevance target. We grade on `val`; no cell here reads the sealed `test` split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])


## Exercise 1 - Cosine nearest words (scoped to the catalog's vocabulary)

A GloVe vector turns a word into a point in a 100-dimensional semantic space, and **cosine similarity** between two such vectors measures how related the words are: `cos(a, b) = (a . b) / (|a| |b|)`, which is 1.0 for identical directions and near 0 for unrelated ones. Because every book embedding is built from these word vectors, the quickest way to *see* the space is to find a word's nearest neighbours.

For the query word, compute the cosine similarity of its GloVe vector to **every** word in the subset's vocabulary and report the 5 nearest (skip the query word itself and any out-of-vocabulary zero row). Meaning-related words rise to the top. Label the result honestly: these are the nearest among **the catalog's own 607-word vocabulary**, not the full-GloVe analogies a 400k-word model would give -- the committed subset is restricted on purpose.


In [ ]:
query_token = "dragon"   # try "ocean", "wizard", "castle" too -- all in the vocabulary

matrix = glove.matrix                       # (608, 100) float32; row i is glove.tokens[i]
norms = np.linalg.norm(matrix, axis=1)      # per-row length; the one OOV row has length 0
norms[norms == 0.0] = 1.0                   # avoid divide-by-zero on the zero (OOV) row
unit = matrix / norms[:, None]              # every row L2-normalized -> cosine is a dot product

query_row = glove.token_index[query_token]  # the query word's row in the matrix
cosines = unit @ unit[query_row]            # cosine of the query against every vocab word

# Rank the vocabulary by cosine (descending), skip the query's own row and any OOV zero row,
# take the 5 nearest TOKENS.
order = np.argsort(-cosines)
nearest_tokens = []
for row in order:
    token = glove.tokens[row]
    if row == query_row or token in glove.missing:
        continue
    nearest_tokens.append(token)
    if len(nearest_tokens) == 5:
        break

query_excluded = query_token not in nearest_tokens
five_neighbours = len(nearest_tokens) == 5
nearest_label = "nearest among the catalog's 607-word vocabulary (not full-GloVe analogies)"
top_cosine = float(cosines[glove.token_index[nearest_tokens[0]]])
meaning_explanation = (
    "Cosine in GloVe space measures the ANGLE between two word vectors, and because GloVe is "
    "trained so that words used in similar contexts point in similar directions, that angle is a "
    "proxy for MEANING. So 'dragon' lands near 'sword', 'monster' and 'demon' even though those "
    "words share no letters with it -- a relationship Unit 3's bag-of-words matching, which can "
    "only see literally shared tokens, is blind to."
)

print(f"{query_token!r} ->", nearest_tokens)
print(f"top cosine {top_cosine:.3f}  ({nearest_label})")

assert query_excluded, "the query word must not be its own neighbour"
assert five_neighbours, "exactly 5 nearest vocabulary words"
assert all(t in glove.token_index and t not in glove.missing for t in nearest_tokens)
assert top_cosine > 0.3, "the nearest word should be clearly meaning-related, not noise"


## Exercise 2 - Build a book and a reader embedding by hand

A book is a **bag of keyword tokens** (the same text Unit 3 indexed). Its **document embedding** is the simplest composition: the **mean** of its in-vocabulary tokens' GloVe vectors, then **L2-normalized** so every book is a unit vector and cosine is a plain dot product. Out-of-vocabulary tokens are skipped; a book with no in-vocabulary token is a safe all-zero vector.

Build one book's embedding **by hand** from its keyword tokens (`glove.vector(token)` returns the vector or `None` when out of vocabulary), then confirm it matches the shipped `book_embedding(tokens, glove)`. Then aggregate a **reader embedding** the same way -- the mean of the embeddings of the books in one warm reader's `seen` set, L2-normalized -- exactly the query the retrieval path scores against the catalog.


In [ ]:
book_id = catalog_ids[0]
tokens = keywords[book_id].split()          # this book's keyword bag as a token list

# --- by hand: mean of the in-vocabulary token vectors, then L2-normalize ---
token_vectors = [glove.vector(t) for t in tokens if glove.vector(t) is not None]
mean_vector = np.mean(np.array(token_vectors), axis=0)
my_embedding = mean_vector / np.linalg.norm(mean_vector)   # L2-normalize

library_embedding = book_embedding(tokens, glove)
embeddings_match = bool(np.allclose(my_embedding, library_embedding, atol=1e-5))
is_unit_length = bool(np.isclose(np.linalg.norm(library_embedding), 1.0))

# --- a reader embedding: the mean of their seen books' embeddings, L2-normalized ---
reader_id = sorted(r for r in seen_by if r not in cold_readers)[0]   # the first warm reader
seen = seen_by[reader_id]
seen_embeddings = np.array([book_embedding(keywords[b].split(), glove) for b in seen])
reader_mean = seen_embeddings.mean(axis=0)
reader_embedding = reader_mean / np.linalg.norm(reader_mean)   # L2-normalize
reader_is_unit_length = bool(np.isclose(np.linalg.norm(reader_embedding), 1.0))

normalize_explanation = (
    "L2-normalizing puts every book and reader on the unit sphere, so a cosine (a dot product of "
    "unit vectors) compares pure DIRECTION -- the meaning the keywords point at -- and nothing "
    "else. Without it, a book with many keywords would have a longer mean vector and could outscore "
    "a book with few keywords on length alone, even if its direction were a worse match."
)

print(f"book {book_id}: hand == library? {embeddings_match}; unit length? {is_unit_length}")
print(f"reader {reader_id}: unit length? {reader_is_unit_length}")

assert embeddings_match, "the hand-built embedding must match the shipped book_embedding"
assert is_unit_length, "book_embedding returns an L2-normalized (unit) vector"
assert reader_is_unit_length, "the reader embedding is L2-normalized too"


## Exercise 3 - Register the semantic path and read the scoreboard honestly

Now wrap the embedding retriever as a first-class path. `SemanticEmbeddingRetrievalPath(keywords, glove)` takes the keyword corpus and the GloVe subset at construction (like `LexicalRetrievalPath`), so `fit(rows, catalog=catalog_ids)` keeps the exact protocol signature and merely precomputes each book's embedding; `retrieve` builds the reader embedding from `context["seen"]` and scores the catalog by cosine. Register it in a `PathRegistry` (it must own the name `semantic-v1`), then score it on `val` beside lexical, the item-item CF and matrix-factorization collaborative paths, and the random floor.

Read the numbers honestly: semantic lands around hit@10 ~0.102 -- clearly above the floor (~0.012, about 8.5x), but **below** lexical (~0.158) and far below item-item CF (~0.252) and MF (~0.276). A content path that clears the floor and loses to lexical here -- *not* a win. It trails lexical because the latent-taste topics are arbitrary slices of a general word pool, so GloVe's meaning-geometry is uncorrelated-by-construction with the taste structure (only the genre banks help); a real corpus would differ. Precision@k and NDCG@k rank the paths in the same order -- a weak path is weak on every lens.


In [ ]:
# Fit the paths once; later exercises REUSE these fitted objects. The MF fit is the only slow one
# (~15 s). Keep this to five fits.
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)       # collaborative filtering
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)   # matrix factorization
floor = RandomRetrievalPath(catalog_ids, seed=SEED)

# Register the semantic path and confirm it owns semantic-v1.
registry = PathRegistry()
registry.register(semantic)
artifact_name = semantic.artifact_name()                 # 'semantic-v1'
registered_as_semantic = artifact_name == "semantic-v1" and "semantic" in registry


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )


semantic_board = val_hit(semantic)          # hit ~0.102, precision ~0.011, ndcg ~0.033
lexical_hit = val_hit(lexical).hit_rate_at_k        # ~0.158
cf_hit = val_hit(item_item).hit_rate_at_k           # ~0.252
mf_hit = val_hit(mf).hit_rate_at_k                  # ~0.276
floor_hit = val_hit(floor).hit_rate_at_k            # ~0.012
semantic_hit = semantic_board.hit_rate_at_k         # ~0.102

clears_floor = semantic_hit > 5 * floor_hit
below_lexical = semantic_hit < lexical_hit
below_collaborative = lexical_hit < cf_hit and lexical_hit < mf_hit
honest_reading = (
    "Semantic clears the random floor by about 8.5x, so GloVe-averaged book vectors do carry real "
    "taste signal -- but it is BELOW lexical (about 65% of its hit@10) and far below both "
    "collaborative paths (item-item CF and MF). It is a content path, not a win. It trails lexical "
    "here because this catalog's latent topics are arbitrary contiguous slices of a general word "
    "pool, so GloVe's meaning-geometry is uncorrelated-by-construction with the taste structure "
    "(only the 12 genre banks carry GloVe-exploitable meaning); on a real catalog's prose dense "
    "semantics would fare differently. Precision@k and NDCG@k order the paths identically -- a weak "
    "path is weak on every lens."
)

print(f"floor     hit@{K} = {floor_hit:.3f}")
print(f"semantic  hit@{K} = {semantic_hit:.3f}  recall {semantic_board.recall_at_k:.3f}  "
      f"precision {semantic_board.precision_at_k:.3f}  ndcg {semantic_board.ndcg_at_k:.4f}")
print(f"lexical   hit@{K} = {lexical_hit:.3f}")
print(f"item-item hit@{K} = {cf_hit:.3f}")
print(f"mf        hit@{K} = {mf_hit:.3f}")

assert registered_as_semantic, "the path must own and register as 'semantic-v1'"
assert clears_floor, "semantic must beat the random floor by >5x"
assert below_lexical, "honest: semantic is below lexical on this data"
assert 0.5 * lexical_hit <= semantic_hit <= 1.1 * lexical_hit, "semantic sits in the measured band"
assert below_collaborative, "lexical (and so semantic) is far below the collaborative CF/MF paths"


## Exercise 4 - Where semantic earns its place: complementarity

A path that loses to lexical head-to-head can still be worth keeping -- if it is *wrong about different books*. That is exactly semantic's case. For every eligible warm reader, retrieve a top-10 from **lexical** and a top-10 from **semantic** and measure the **overlap** (the fraction of the 10 slots the two lists share); average it across readers. The mean overlap is only about **0.22** -- the two paths agree on roughly two books in ten and surface eight different ones.

Then make it concrete for one reader: list the books semantic puts in its top-10 that lexical does **not**, and look at one such book's keyword bag. Because semantic matches on GloVe *meaning*, it can relate books that share no surface token with the reader's history -- precisely what a bag-of-words path cannot do.


In [ ]:
# Mean lexical-vs-semantic top-10 overlap across eligible warm readers.
overlaps = []
for rid in sorted(val_pos_by):
    seen_r = seen_by[rid]
    relevant = val_pos_by[rid] - seen_r
    if rid in cold_readers or not relevant:
        continue
    sem_ids = [c.item_id for c in rank(semantic.retrieve(rid, {"seen": seen_r}, K), n=K, exclude=seen_r)]
    lex_ids = [c.item_id for c in rank(lexical.retrieve(rid, {"seen": seen_r}, K), n=K, exclude=seen_r)]
    if sem_ids and lex_ids:
        overlaps.append(len(set(sem_ids) & set(lex_ids)) / K)

mean_overlap = float(np.mean(overlaps))     # about 0.22
overlap_is_low = 0.0 < mean_overlap < 0.4   # mostly DIFFERENT books

# One reader, concretely: the books semantic finds that lexical misses.
example_reader = sorted(r for r in val_pos_by if r not in cold_readers and (val_pos_by[r] - seen_by[r]))[0]
ex_seen = seen_by[example_reader]
ex_sem = [c.item_id for c in rank(semantic.retrieve(example_reader, {"seen": ex_seen}, K), n=K, exclude=ex_seen)]
ex_lex = [c.item_id for c in rank(lexical.retrieve(example_reader, {"seen": ex_seen}, K), n=K, exclude=ex_seen)]
semantic_only = sorted(set(ex_sem) - set(ex_lex))   # books only semantic surfaces
semantic_finds_extra = len(semantic_only) >= 1
example_keyword_bag = keywords[semantic_only[0]]    # a meaning-related book, not a token match

complementarity_explanation = (
    "A LOW overlap -- not a high hit@10 -- is what makes semantic a promising blend ingredient. "
    "Because it agrees with lexical on only about two books in ten, it is wrong about DIFFERENT "
    "books: it matches on GloVe meaning where lexical needs literally shared tokens, so it surfaces "
    "meaning-related candidates lexical never sees. A blend can gain from a weak-but-different path, "
    "where a second copy of a strong-but-redundant path would add nothing."
)

print(f"mean lexical-vs-semantic top-10 overlap = {mean_overlap:.3f}  (low={overlap_is_low})")
print(f"reader {example_reader}: {len(semantic_only)} semantic-only books, e.g. {semantic_only[0]}")

assert overlap_is_low, "the two paths mostly surface DIFFERENT books (overlap well below 0.4)"
assert semantic_finds_extra, "semantic must surface at least one book lexical misses"
assert isinstance(example_keyword_bag, str) and example_keyword_bag, "the example book has a keyword bag"


## Exercise 5 - Challenge: max-similarity pooling vs the mean

Mean pooling blurs a reader's whole history into **one** centroid vector, which can wash out a reader with several distinct interests. An alternative, **max-similarity-over-seen**, scores each candidate book by its single *best* cosine to **any** one of the reader's seen books -- so a candidate that strongly matches even one past favourite ranks high. Measured on `val`, max-sim pooling scores hit@10 ~0.142, noticeably **above** mean pooling's ~0.102 (a documented alternative, not the shipped default).

Using the fitted path's precomputed book-embedding matrix (`semantic.artifact()["embeddings"]`, with `["item_ids"]` as the row order -- every row is already unit-length), build the **max-sim** top-10 for one warm reader and compare it to the mean-pooled top-10 from `semantic.retrieve`. They differ.


In [ ]:
art = semantic.artifact()
book_matrix = np.array(art["embeddings"])     # (n_books, 100); each row L2-normalized (or zero)
item_ids = list(art["item_ids"])              # the row order of book_matrix
row_of = {iid: i for i, iid in enumerate(item_ids)}

stretch_reader = sorted(r for r in seen_by if r not in cold_readers)[0]
stretch_seen = seen_by[stretch_reader]
seen_rows = [row_of[b] for b in stretch_seen if b in row_of]

# max-sim-over-seen: each book's score is its BEST cosine to any one seen book.
sim_to_seen = book_matrix @ book_matrix[seen_rows].T   # (n_books, n_seen)
max_sim = sim_to_seen.max(axis=1)                      # best match per book
order = np.argsort(-max_sim)
maxsim_top = [item_ids[j] for j in order if item_ids[j] not in stretch_seen][:K]

# mean pooling (the shipped default) for the SAME reader, via the path.
mean_top = [c.item_id for c in rank(
    semantic.retrieve(stretch_reader, {"seen": stretch_seen}, K), n=K, exclude=stretch_seen
)]

poolings_differ = set(maxsim_top) != set(mean_top)
# Measured on val (seed 0, k=10, 500 readers): mean pooling ~0.102, max-sim-over-seen ~0.142.
maxsim_explanation = (
    "A MAX over the seen books keeps any candidate that strongly matches even ONE past favourite, "
    "so a multi-interest reader's every taste stays represented; the MEAN averages all those tastes "
    "into one centroid, which can land between interests and match none of them well -- that is why "
    "max-sim scores higher here (~0.142 vs ~0.102). The cost of max-sim is that it ignores "
    "CONSISTENCY: a book that matches one favourite but clashes with the rest of the history still "
    "ranks high, where the mean would have discounted it."
)

print(f"max-sim top-10 differs from mean top-10? {poolings_differ}")
print(f"max-sim: {maxsim_top}")
print(f"mean   : {mean_top}")

assert poolings_differ, "max-sim and mean pooling give different top-10 lists"
assert len(maxsim_top) == K and len(set(maxsim_top)) == K, "max-sim yields K distinct books"
assert not (set(maxsim_top) & stretch_seen), "max-sim never recommends an already-seen book"


## Exercise 6 - Challenge: add semantic to the Unit 6 blend

This is the quantitative complementarity test. Reuse the Unit 6 **pinned** blend (per-path pool 30; weights `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}`) and measure its hit@10 and catalog coverage on `val`. Then add the semantic path at a modest weight (0.5) and measure again. Even though semantic *loses* to lexical on its own (Exercise 3), it lifts **both** numbers here -- hit@10 from ~0.306 to ~0.316 and coverage from ~0.333 to ~0.353 -- because it pulls in meaning-related books the other paths miss.

Fit the one remaining path (popularity), reuse the Exercise-3 fits, and compute the two blended scoreboards. This is the complementarity claim made quantitative (a path below lexical still helps the blend) -- but it is a result at **this pinned config**: the semantic weight is chosen in advance, not swept on `val`, so the lift is not a validation-sweep maximum reported as expected performance.


In [ ]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)   # the one path not yet fitted

WEIGHTS_U6 = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}   # the Unit 6 pinned weights
POOL = 30                                                                        # the Unit 6 pinned pool
paths_without = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}
paths_with = dict(paths_without, semantic=semantic)
weights_with = dict(WEIGHTS_U6, semantic=0.5)   # add semantic at a modest, pre-chosen weight


def blended(path_map, weight_map):
    return run_blended_scoreboard(
        path_map, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
        weights=weight_map, cold_readers=cold_readers,
    )


without_semantic = blended(paths_without, WEIGHTS_U6)   # hit ~0.306 / cov ~0.333
with_semantic = blended(paths_with, weights_with)       # hit ~0.316 / cov ~0.353

hit_rises = with_semantic.hit_rate_at_k > without_semantic.hit_rate_at_k
coverage_rises = with_semantic.catalog_coverage > without_semantic.catalog_coverage
blend_reading = (
    "Adding the semantic path -- which LOSES to lexical on its own -- lifts BOTH the blend's hit@10 "
    "(about 0.306 -> 0.316) and its catalog coverage (about 0.333 -> 0.353), because it contributes "
    "meaning-related books the other paths miss: complementarity made quantitative, the point "
    "Exercise 4's low overlap predicted. But this is a result at the Unit 6 PINNED config, with the "
    "semantic weight pre-chosen at 0.5 rather than swept on val, so the lift is an honest "
    "ablation -- not a validation-sweep maximum dressed up as expected performance."
)

print(f"without semantic: hit@{K} = {without_semantic.hit_rate_at_k:.3f}  "
      f"coverage = {without_semantic.catalog_coverage:.3f}")
print(f"with semantic   : hit@{K} = {with_semantic.hit_rate_at_k:.3f}  "
      f"coverage = {with_semantic.catalog_coverage:.3f}")

assert hit_rises, "adding semantic to the pinned blend raises hit@10"
assert coverage_rises, "adding semantic to the pinned blend raises catalog coverage"
assert without_semantic.hit_rate_at_k > semantic_hit, "the blend already beats semantic alone"


## Exercise 7 - Challenge: the dot-product retriever that connects three units

Semantic retrieval, Unit 5's matrix factorization, and Unit 8's coming two-tower are the **same retriever** wearing different clothes: each represents a reader and a book as vectors in one shared space and scores them by a **dot product** (a cosine, once the vectors are unit-length). The only thing that changes is *where the vectors come from* -- GloVe's are **pretrained** (fixed, from Wikipedia text), MF's are **learned** from the interaction log, and the two-tower's are **learned by a neural network**.

Make the equivalence concrete. Rebuild one reader's semantic score **by hand** as a dot product -- the L2-normalized mean of their seen books' embedding rows, dotted against the book matrix -- and confirm the top book and its score match what `semantic.retrieve` returns. Then look at the MF factors (`mf.artifact()`): its reader matrix `P` and item matrix `Q` live in the same factor dimension, and MF scores reader `r` against book `i` as `P[r] . Q[i]` -- the identical operation on *learned* vectors.


In [ ]:
# Reuse Exercise 5's book_matrix / item_ids / row_of (the path's own precomputed embeddings).
bridge_reader = sorted(r for r in seen_by if r not in cold_readers)[0]
bridge_seen = seen_by[bridge_reader]
bridge_rows = [row_of[b] for b in bridge_seen if b in row_of]

# By hand: reader vector = L2-normalized mean of seen rows; score every book by a dot product.
reader_vec = book_matrix[bridge_rows].mean(axis=0)
reader_vec_unit = reader_vec / np.linalg.norm(reader_vec)
hand_scores = book_matrix @ reader_vec_unit   # cosine of every book vs the reader

# The top unseen book by the hand dot product.
hand_order = np.argsort(-hand_scores)
hand_top_id = next(item_ids[j] for j in hand_order if item_ids[j] not in bridge_seen)
hand_top_score = float(hand_scores[row_of[hand_top_id]])

# What the path returns for the same reader.
path_top = rank(semantic.retrieve(bridge_reader, {"seen": bridge_seen}, K), n=K, exclude=bridge_seen)[0]
matches_path = hand_top_id == path_top.item_id and bool(np.isclose(hand_top_score, path_top.score, atol=1e-5))

# MF does the SAME dot product on LEARNED factors: score(r, i) = P[r] . Q[i].
mf_art = mf.artifact()
P = np.array(mf_art["P"])   # reader factors (readers x n_factors)
Q = np.array(mf_art["Q"])   # item   factors (items   x n_factors)
same_factor_space = P.shape[1] == Q.shape[1]   # reader and item vectors share one space

bridge_explanation = (
    "The hand dot product reproduces the path's top book and score exactly, because retrieval IS a "
    "dot product of the L2-normalized reader vector against the unit-length book rows -- a cosine. "
    "MF scores reader r against book i as P[r] . Q[i], the identical operation, and P and Q share "
    "one factor dimension so they live in the same space. Semantic (pretrained GloVe here), MF "
    "(factors LEARNED from the log, Unit 5), and Unit 8's two-tower (vectors LEARNED by a neural "
    "network) are therefore all the same dot-product retriever over reader/book embeddings -- only "
    "the SOURCE of the vectors differs."
)

print(f"hand top book {hand_top_id} (score {hand_top_score:.4f}) == path top {path_top.item_id} "
      f"(score {path_top.score:.4f})? {matches_path}")
print(f"MF factor space: P {P.shape}, Q {Q.shape}; shared dimension? {same_factor_space}")

assert matches_path, "the by-hand dot product reproduces the path's top book and score"
assert same_factor_space, "MF reader and item factors share one dot-product space"
assert np.isclose(np.linalg.norm(reader_vec_unit), 1.0), "the reader query is a unit vector"
